# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies


## Task 2: Environment Variables

We'll want to set both our OpenAI API key and our LangSmith environment variables.

In [19]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [20]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [21]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE7 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [64]:
from langchain_tavily import TavilySearch  # ← Changed import
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearch(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [65]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [66]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

---

The model determines which tool to used based on an analysis of the tool's description and the query. From my conversation with Claude, I understand that when the tools are bound to the model, they are converted into a special format and included in the model's context. The LLM then uses semantic similarity to extract the intent, domain, and required information type of the user query and compares that to each tool's description.

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [67]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [68]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [69]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [70]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [71]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [72]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [73]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

---

There is no specific limit to how many times we can cycle, but it would seem that you would want to set limits in order to guardrail and prevent infinite loops, expensive API usage, and unnecessary cycles.

We could impose a limit to the number of cycles using several methods:
1. Implement a length limit to state["messages"]
2. Build in a recursion limit in graph.invoke(inputs, {"recursion_limit": 10}) or graph.astream(inputs, {"recursion_limit": 10})
3. Add a cycle counter to state
4. Implement a time based limit

## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [74]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="Who is the current captain of the Winnipeg Jets?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3PgdqHWPJW9hmfDeUKpMDekZ', 'function': {'arguments': '{"query":"current captain of the Winnipeg Jets"}', 'name': 'tavily_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 873, 'total_tokens': 894, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BteWJeEffAkuTB2vVCMsOxlNjvV1m', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--9a8a283a-1596-47ce-b5f2-442a7904bafe-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current captain of the Winnipeg Jets'}, 'id': 'call_3PgdqHWPJW9hmfDeUKpMDekZ', 'type': 'tool_call'}], usage_metad

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [79]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_fMgOMm9Zqm9wYXKYh00UDPPo', 'function': {'arguments': '{"query": "QLoRA"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_oOF8AbtK5zTWiYxNBIQ97mhG', 'function': {'arguments': '{"query": "latest Tweet by author"}', 'name': 'tavily_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 889, 'total_tokens': 940, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BteXiRShsJCuHiyd2aBeHjmiIkH2Y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--68117369-7780-4b8c-ac65-9566b1d0a547-0', tool_calls=[{'name': 'arxiv', 'args': {'query': 'QLoRA'}

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

0. START
1. Agent Node: Process user propmt and determine that the next step requires the arxiv tool call with the query 'QLoRA'.
2. Action Node [arxiv]: Utilize tool arxiv to extract the QLoRA paper.
3. Agent Node: Process arxiv tool output to extract the 4 author names and determine that the next step requires the tavily tool call with the query '{AUTHOR} latest tweet'.
4. Action Node [tavily]: Utilize tool tavily to search Twitter for the twitter account of each AUTHOR and extract the latest tweet. This is run in parallel for each author.
5. Agent Node: Process tavily tool output to return to the user the latest tweets from the authors of the QLoRA paper.
6. END

# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [80]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["question"])]}

def parse_output(input_state):
  return input_state["messages"][-1].content

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

In [81]:
agent_chain_with_formatting.invoke({"question" : "What is RAG?"})

'RAG, or Retrieval-Augmented Generation, is an AI framework that enhances the output of large language models (LLMs) by incorporating an external knowledge base or data source before generating a response. It involves an information retrieval component that first pulls relevant data from external sources based on user input, and then the language model uses this retrieved information to produce more accurate, reliable, and up-to-date responses. This approach helps reduce hallucinations in AI responses and ensures that the generated content is grounded in authoritative information.'

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions.

In [82]:
questions = [
    "Who won the 2024 Nobel Prize in Physics?",
    "What are the key innovations in the BERT paper?",
    "What is the current stock price of NVIDIA?",
    "Find recent papers on quantum machine learning algorithms",
    "Who are the authors of the original Transformer paper 'Attention is All You Need'?",
    "What is the weather forecast for San Francisco tomorrow?"
]

answers = [
    {"must_mention": ["Hinton", "Hopfield"]},
    {"must_mention": ["bidirectional", "pre-training", "masked"]},
    {"must_mention": ["NVIDIA", "stock", "price"]},
    {"must_mention": ["quantum", "machine learning", "algorithm"]},
    {"must_mention": ["Vaswani", "Shazeer", "Parmar"]},
    {"must_mention": ["San Francisco", "weather", "forecast"]}
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [83]:
from langsmith import Client

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the QLoRA Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=[{"question" : q} for q in questions],
    outputs=answers,
    dataset_id=dataset.id,
)

{'example_ids': ['b1d3f9b8-3991-4cc6-a551-ca537d7689d7',
  '44c5d1bc-777c-485a-85a5-c3277086c622',
  '334393b4-b1b8-4238-8ac2-2e0115164008',
  '00002231-225d-4aa1-88ef-4841df6b6e81',
  'e51cfdf7-b2d1-4e89-a32b-0c821f041227',
  'a9d185d7-1c10-495a-b78b-00d4e6883dc9'],
 'count': 6}

#### ❓ Question #3:

How are the correct answers associated with the questions?
> NOTE: Feel free to indicate if this is problematic or not

---

- The correct answers are associated with the questions by index position. 
- This is problematic because it creates an order dependency which can be prone to error, as well as a length mismatch in the event that there are more or less questions than answers. 
- The correct answers should instead be paired with the question directly, in a list of dicts for example.

### Task 2: Adding Evaluators

Now we can add a custom evaluator to see if our responses contain the expected information.

We'll be using a fairly naive exact-match process to determine if our response contains specific strings.

In [84]:
from langsmith.evaluation import EvaluationResult, run_evaluator

@run_evaluator
def must_mention(run, example) -> EvaluationResult:
    prediction = run.outputs.get("output") or ""
    required = example.outputs.get("must_mention") or []
    score = all(phrase in prediction for phrase in required)
    return EvaluationResult(key="must_mention", score=score)

#### ❓ Question #4:

What are some ways you could improve this metric as-is?
> NOTE: Alternatively you can suggest where gaps exist in this method.

---

- Case-insensitive matching
- Fuzzy match
- Semantic similarity using embeddings

The main gap here is in the exact match, which may differ slightly by upper vs lowercase, how it is placed within a sentence, etc. However even an exact match may yield an answer that does not answer the question, even if the keywords are perfectly matched.


Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [85]:
experiment_results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset_name,
    evaluators=[must_mention],
    experiment_prefix=f"Search Pipeline - Evaluation - {uuid4().hex[0:4]}",
    metadata={"version": "1.0.0"},
)

View the evaluation results for experiment: 'Search Pipeline - Evaluation - 47c9-86e63ad7' at:
https://smith.langchain.com/o/4e8d6936-58a0-4ff9-bfae-6484988e5784/datasets/6be02ac6-3e67-4b2a-b407-9f3b559f9bea/compare?selectedSessions=5ffd843b-34c6-4966-a784-9aea166e114f




0it [00:00, ?it/s]

In [86]:
experiment_results

<ExperimentResults Search Pipeline - Evaluation - 47c9-86e63ad7>

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [87]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #5:

Please write markdown for the following cells to explain what each is doing.

##### Build a new LangGraph that includes a helpfulness check
- Create an agent node for LLM calls
- Create an action node for tool calls

In [88]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### Set the graph entry point to be the agent node

In [90]:
graph_with_helpfulness_check.set_entry_point("agent")

##### Define routing logic to determine how the graph should proceed

In [92]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

#### 🏗️ Activity #4:

Please write what is happening in our `tool_call_or_helpful` function!

---

- If the last message is a tool call, then go to the action node.
- If the # of messages is greater than 10, then END the agent.
- If no tools are needed and we have not met the max # of messages, then evaluate if the response is helpful. If so, END the agent. If not, continue the evaluation cycle.

##### Add conditional routing based on the tool_call_or_helpful function
- Three options: continue (i.e. go to the agent node), action (i.e. go to the action node), or end (i.e. END the agent)

In [93]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### Add a direct edge from the action node back to agent node.
- This is ALWAYS executed, unlike the conditional edge which can go to one of 3 nodes.

In [94]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### Compile the LangGraph to create an executable agent

In [95]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### Test the helpfulness agent with a complex, multi-part question

In [96]:
inputs = {"messages" : [HumanMessage(content="Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_23CxBnYFqTbfWsfXUmRydvyi', 'function': {'arguments': '{"query": "LoRA machine learning"}', 'name': 'tavily_search'}, 'type': 'function'}, {'id': 'call_fFYwpMCKoXcOxrTBeUj0TBA3', 'function': {'arguments': '{"query": "Tim Dettmers"}', 'name': 'tavily_search'}, 'type': 'function'}, {'id': 'call_dcNB805OLoQBJ988h6kcg1oz', 'function': {'arguments': '{"query": "Attention in machine learning"}', 'name': 'tavily_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 888, 'total_tokens': 961, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-Btel1BldoGKyXHmLFZ4MHFiqL5TCL', 'service_t

### Task 4: LangGraph for the "Patterns" of GenAI

Let's ask our system about the 4 patterns of Generative AI:

1. Prompt Engineering
2. RAG
3. Fine-tuning
4. Agents

In [97]:
patterns = ["prompt engineering", "RAG", "fine-tuning", "LLM-based agents"]

In [98]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

Prompt engineering is the process of designing and refining prompts to effectively communicate with AI language models, such as GPT, to obtain desired responses. It involves crafting specific, clear, and contextually appropriate prompts to guide the AI's output, often requiring an understanding of how the model interprets language and context.

Prompt engineering has become increasingly prominent with the rise of large language models (LLMs) like GPT-3 and GPT-4, which can generate human-like text based on the prompts they receive. As these models gained widespread adoption, the importance of prompt engineering as a skill and discipline grew, leading to the emergence of best practices, techniques, and even dedicated communities focused on optimizing prompts.

The concept of prompt engineering started gaining significant attention around 2020-2021, coinciding with the release and popularization of OpenAI's GPT-3 in mid-2020. The term and its practices became more mainstream as users and